In [7]:
%pip install ultralytics==8.4.189

# Настройка сохранения датасета, настроек эксперимента и обученных весов вне временной среды Colab.

In [8]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/industrial-vision")
DATASET_DIR = PROJECT_DIR / "datasets" / "ppe"
RUNS_DIR = PROJECT_DIR / "runs"

DATASET_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Папка проекта:", PROJECT_DIR)
print("Папка датасета:", DATASET_DIR)
print("Папка результатов:", RUNS_DIR)

Папка проекта: /content/drive/MyDrive/industrial-vision
Папка датасета: /content/drive/MyDrive/industrial-vision/datasets/ppe
Папка результатов: /content/drive/MyDrive/industrial-vision/runs


# скачивание архива Connstruction-PPE

In [10]:
import requests
import zipfile

DATASET_URL = (
    "https://github.com/ultralytics/assets/releases/download/"
    "v0.0.0/construction-ppe.zip"
)

ARCHIVE_PATH = PROJECT_DIR / "construction-ppe.zip"

if not ARCHIVE_PATH.exists():
    response = requests.get(DATASET_URL, stream=True)
    response.raise_for_status()

    with open(ARCHIVE_PATH, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

print("Размер архива:", round(ARCHIVE_PATH.stat().st_size / (1024 ** 2), 1), "МБ")

with zipfile.ZipFile(ARCHIVE_PATH, "r") as archive:
    print("Количество файлов в архиве:", len(archive.namelist()))

Размер архива: 617.7 МБ
Количество файлов в архиве: 2004


In [11]:
import zipfile

DATA_ROOT = DATASET_DIR / "construction-ppe"

if DATA_ROOT.exists():
    raise FileExistsError(
        f"Датасет уже распакован: {DATA_ROOT}. "
        "Повторная распаковка запрещена."
    )

with zipfile.ZipFile(ARCHIVE_PATH, "r") as archive:
    archive.extractall(DATASET_DIR)

print("Датасет распакован:", DATA_ROOT)

Датасет распакован: /content/drive/MyDrive/industrial-vision/datasets/ppe/construction-ppe


# Инофрмация о датасете

In [12]:
import yaml

DATA_YAML = DATA_ROOT / "data.yaml"

with open(DATA_YAML, "r", encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)

print("Настройки датасета:")
print(yaml.dump(dataset_config, allow_unicode=True, sort_keys=False))

print("Количество изображений:")
for split in ("train", "val", "test"):
    image_dir = DATA_ROOT / "images" / split

    if image_dir.exists():
        count = sum(
            1 for path in image_dir.iterdir()
            if path.suffix.lower() in (".jpg", ".jpeg", ".png")
        )
        print(f"{split}: {count}")
    else:
        print(f"{split}: папка отсутствует")

Настройки датасета:
path: construction-ppe
train: images/train
val: images/val
names:
  0: helmet
  1: gloves
  2: vest
  3: boots
  4: goggles
  5: Person
  6: no_helmet
  7: no_goggle
  8: no_gloves
  9: no_boots
  10: forklift
  11: no_vest

Количество изображений:
train: 800
val: 200
test: папка отсутствует


In [13]:
README_PATH = DATA_ROOT / "README.md"

with open(README_PATH, "r", encoding="utf-8") as file:
    print(file.read())

print("\nПримеры файлов разметки:")

LABELS_DIR = DATA_ROOT / "labels" / "train"

label_files = list(LABELS_DIR.glob("*.txt"))

for label_file in label_files[:3]:
    print(f"\n{label_file.name}:")
    print(label_file.read_text(encoding="utf-8"))

# Ultralytics Construction-PPE

Ultralytics Construction-PPE is an original dataset created and open-sourced in November 2026 by the Ultralytics team. It combines human-collected and generated images, with expert human annotation and AI auto-annotation completed in November 2026 by the Ultralytics team.

Images and annotations are licensed under Creative Commons Attribution 4.0 International (CC BY 4.0): https://creativecommons.org/licenses/by/4.0/.

Attribution: Ultralytics Construction-PPE Dataset, Ultralytics team, November 2026, CC BY 4.0. Link to the dataset and license and indicate changes when redistributing or adapting it.

- Platform: https://platform.ultralytics.com/ultralytics/datasets/construction-ppe
- Documentation: https://docs.ultralytics.com/datasets/detect/construction-ppe
- Images: 1,000 (800 train, 200 validation; no separate test split).
- Classes: 12; IDs and names are defined in data.yaml.
- Annotations: 14,863 YOLO detection boxes, one class ID and normalized ce

# 70-15-15

In [14]:

import random
import shutil
import yaml

# Исходный датасет
SOURCE_ROOT = DATA_ROOT

# Новый датасет с распределением 70/15/15
OUTPUT_ROOT = DATASET_DIR / "construction-ppe-70-15-15"

# Собираем изображения и соответствующие им файлы разметки
samples = []

for split in ("train", "val"):
    image_dir = SOURCE_ROOT / "images" / split
    label_dir = SOURCE_ROOT / "labels" / split

    for image_path in image_dir.iterdir():
        if image_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
            continue

        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            raise FileNotFoundError(f"Нет разметки: {label_path}")

        samples.append((image_path, label_path))

# Проверяем количество изображений
print("Всего изображений:", len(samples))

# Перемешиваем изображения в случайном, но воспроизводимом порядке
random.Random(42).shuffle(samples)

# Разделяем на три части
train_end = int(len(samples) * 0.70)
val_end = int(len(samples) * 0.85)

splits = {
    "train": samples[:train_end],
    "val": samples[train_end:val_end],
    "test": samples[val_end:]
}

# Создаём папки и копируем изображения вместе с разметкой
if OUTPUT_ROOT.exists():
    raise FileExistsError(
        f"Папка уже существует: {OUTPUT_ROOT}. "
        "Не перезаписываем существующий датасет."
    )

for split, files in splits.items():
    image_dir = OUTPUT_ROOT / "images" / split
    label_dir = OUTPUT_ROOT / "labels" / split

    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)

    for image_path, label_path in files:
        shutil.copy2(image_path, image_dir / image_path.name)
        shutil.copy2(label_path, label_dir / label_path.name)

# Создаём конфигурацию нового датасета
new_config = {
    "path": str(OUTPUT_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": dataset_config["names"]
}

with open(OUTPUT_ROOT / "data.yaml", "w", encoding="utf-8") as file:
    yaml.safe_dump(
        new_config,
        file,
        allow_unicode=True,
        sort_keys=False
    )

# Выводим результат
print("\nНовое распределение:")
for split, files in splits.items():
    print(f"{split}: {len(files)} изображений")

print("\nНовый датасет:", OUTPUT_ROOT)


Всего изображений: 1000

Новое распределение:
train: 700 изображений
val: 150 изображений
test: 150 изображений

Новый датасет: /content/drive/MyDrive/industrial-vision/datasets/ppe/construction-ppe-70-15-15


# Изменение разметки

In [15]:

# Исходные ID нужных классов и их новые ID
CLASS_MAPPING = {
    0: 0,   # helmet
    6: 1,   # no_helmet
    2: 2,   # vest
    11: 3   # no_vest
}

CLASS_NAMES = {
    0: "helmet",
    1: "no_helmet",
    2: "vest",
    3: "no_vest"
}

# Перерабатываем разметку во всех трёх выборках
for split in ("train", "val", "test"):
    label_dir = OUTPUT_ROOT / "labels" / split

    for label_path in label_dir.glob("*.txt"):
        new_lines = []

        with open(label_path, "r", encoding="utf-8") as file:
            for line in file:
                values = line.split()

                if not values:
                    continue

                old_class_id = int(values[0])

                if old_class_id in CLASS_MAPPING:
                    values[0] = str(CLASS_MAPPING[old_class_id])
                    new_lines.append(" ".join(values))

        with open(label_path, "w", encoding="utf-8") as file:
            file.write("\n".join(new_lines))

# Обновляем конфигурацию датасета
new_config["names"] = CLASS_NAMES

with open(OUTPUT_ROOT / "data.yaml", "w", encoding="utf-8") as file:
    yaml.safe_dump(
        new_config,
        file,
        allow_unicode=True,
        sort_keys=False
    )

print("Разметка обновлена.")
print("Классы модели:", CLASS_NAMES)


Разметка обновлена.
Классы модели: {0: 'helmet', 1: 'no_helmet', 2: 'vest', 3: 'no_vest'}


In [16]:

from collections import Counter

class_counts = {
    split: Counter()
    for split in ("train", "val", "test")
}

for split in ("train", "val", "test"):
    label_dir = OUTPUT_ROOT / "labels" / split

    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r", encoding="utf-8") as file:
            for line in file:
                values = line.split()

                if values:
                    class_id = int(values[0])
                    class_counts[split][class_id] += 1

print("Количество объектов каждого класса:\n")

for class_id, class_name in CLASS_NAMES.items():
    print(f"{class_name}:")
    for split in ("train", "val", "test"):
        print(f"  {split}: {class_counts[split][class_id]}")
    total = sum(class_counts[split][class_id] for split in class_counts)
    print(f"  Всего: {total}\n")


Количество объектов каждого класса:

helmet:
  train: 1078
  val: 252
  test: 246
  Всего: 1576

no_helmet:
  train: 581
  val: 112
  test: 120
  Всего: 813

vest:
  train: 1243
  val: 290
  test: 283
  Всего: 1816

no_vest:
  train: 409
  val: 73
  test: 85
  Всего: 567



In [18]:

import yaml

DATA_YAML = OUTPUT_ROOT / "data.yaml"

with open(DATA_YAML, "r", encoding="utf-8") as file:
    print(yaml.dump(
        yaml.safe_load(file),
        allow_unicode=True,
        sort_keys=False
    ))


path: /content/drive/MyDrive/industrial-vision/datasets/ppe/construction-ppe-70-15-15
train: images/train
val: images/val
test: images/test
names:
  0: helmet
  1: no_helmet
  2: vest
  3: no_vest



# Обучение второй модели

In [19]:

from ultralytics import YOLO

# Загружаем предобученную модель
model = YOLO("yolo11n.pt")

# Запускаем обучение
results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=15,
    project=str(RUNS_DIR),
    name="ppe_yolo11n",
    exist_ok=False
)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
New https://pypi.org/project/ultralytics/8.4.208 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/industrial-vision/datasets/ppe/construction-ppe-70-15-15/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1

# Проверка на тестовой выборке

In [20]:
from ultralytics import YOLO

model = YOLO(RUNS_DIR / "ppe_yolo11n" / "weights" / "best.pt")

results = model.val(
    data=str(DATA_YAML),
    split="test",
    device=0
)

Ultralytics 8.4.189 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,597,780 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 7.4±15.1 ms, read: 216.6±175.1 MB/s, size: 760.7 KB)
val: Scanning /content/drive/MyDrive/industrial-vision/datasets/ppe/construction-ppe-70-15-15/labels/test... 150 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 221.5it/s 0.7s
val: New cache created: /content/drive/MyDrive/industrial-vision/datasets/ppe/construction-ppe-70-15-15/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.5it/s 6.8s
                   all        150        734      0.885      0.715      0.784      0.585
                helmet        112        246       0.92       0.85      0.903      0.723
             no_helmet         63        120      0.862      0.468      0.571      0.347
                  vest        120        283

# Сохранение результатов

In [21]:
import shutil

TEST_RESULTS_DIR = Path("/content/runs/detect/val")
SAVED_TEST_DIR = RUNS_DIR / "ppe_yolo11n_test"

if not TEST_RESULTS_DIR.exists():
    raise FileNotFoundError("Папка с результатами тестирования не найдена")

if SAVED_TEST_DIR.exists():
    raise FileExistsError("Папка для сохранения результатов уже существует")

shutil.copytree(TEST_RESULTS_DIR, SAVED_TEST_DIR)

print("Результаты тестирования сохранены:", SAVED_TEST_DIR)

Результаты тестирования сохранены: /content/drive/MyDrive/industrial-vision/runs/ppe_yolo11n_test
